# Demo 2 — Practical Demos Notebook
This notebook walks through **each pattern** from Lesson 2 in the same order:
1) Compact CoT (checklist) → 2) Label→Evidence→Verdict → 3) Self-Ask → 4) Contrastive CoT  
5) Plan‑Then‑Solve → 6) Assumptions & Risk → 7) Lightweight Verifier → 8) ReAct loop (glue)

> **Note:** We use small, deterministic *mock* generators so this runs offline. If you want to plug in a real LLM later, replace `mock_model()`.


In [1]:
from typing import List, Dict, Any, Tuple
import json, re

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))

def bullets_from_text(text: str) -> List[str]:
    lines = [ln.strip(" •-*\t") for ln in text.strip().splitlines() if ln.strip()]
    return [ln for ln in lines if ln]

def verify_bullets_count_and_limit(text: str, expected_count: int, max_words_total: int) -> Tuple[bool, Dict[str, Any]]:
    bs = bullets_from_text(text)
    total_words = word_count(" ".join(bs))
    return (len(bs) == expected_count and total_words <= max_words_total,
            {"bullets": len(bs), "total_words": total_words})

def verify_exec_tone(text: str) -> bool:
    banned = {"embedding","vector","matrix","tokenizer","perplexity","latency ms"}
    t = text.lower()
    return not any(b in t for b in banned)

def truncate_words(text: str, max_words: int) -> str:
    words = re.findall(r"\S+", text)
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])

def mock_model(prompt: str) -> str:
    p = prompt.lower()
    if "compact cot" in p or "checklist" in p:
        return "\n".join([
            "Matches requested schema? Yes",
            "Word limit respected? Yes",
            "Evidence cited when required? Yes"
        ])
    if "label → evidence → verdict" in p or "verdict first" in p:
        return "\n".join([
            "Verdict: Supported",
            "Evidence: “Policy requires review within 2 weeks.” (para 2)",
            "Evidence: “Applies to onboarding and quarterly audits.” (para 3)"
        ])
    if "self-ask" in p or "sub-questions" in p:
        return "\n".join([
            "Q1: Audience and tone? A1: Non-technical execs; plain, concise.",
            "Q2: What is retrieval? A2: Finds relevant info to answer.",
            "Q3: Why it helps? A3: Faster, fewer mistakes.",
            "Final: Four bullets ≤60 words, exec tone."
        ])
    if "contrastive" in p and "options" in p:
        return "\n".join([
            "A) Retrieval is like a smart search that shows only what matters.",
            "B) Retrieval brings the **right policy** to your question in seconds.",
            "C) Retrieval reduces noise by filtering to approved sources only.",
            "Pick: B — Why: specific, executive-relevant, action-oriented."
        ])
    if "plan-then-solve" in p or "micro-plan" in p:
        return "\n".join([
            "Plan: define in one line → pick two benefits → add one example.",
            "Output: bullets that follow the plan, ≤ word limit."
        ])
    if "assumption" in p:
        return "Assumptions: access to approved sources; users follow results. Risk: policy updates mid-cycle — Mitigation: version gate."
    if "verifier rule" in p:
        return "Pass: JSON valid; keys present; words within limit."
    if "react loop" in p:
        return "Reason → Act → Observe → Repeat → Final → Verify"
    return "OK"


## 1) Compact CoT (checklist)
**Goal:** Make reasoning *visible* and *auditable* with 3–5 **binary** checks (≤50 words), then return the final answer separately.

In [2]:
task = "Summarize a 2-paragraph policy for execs in 4 bullets (≤60 words)."
prompt = f"Use compact CoT checklist tied to the task. {task}\nReturn 3–5 Yes/No checks only."
checks = mock_model("compact cot checklist")
print("Checklist:\n" + checks)


Checklist:
Matches requested schema? Yes
Word limit respected? Yes
Evidence cited when required? Yes


## 2) Label → Evidence → Verdict (auditable judgments)
**Goal:** Put the **verdict first**, then **1–3 short evidence lines** tied to the provided text (≤50 words).

In [3]:
doc = "Para2: Policy requires review within 2 weeks. Para3: Applies to onboarding and audits."
resp = mock_model("verdict first with label → evidence → verdict")
print(resp)
# Quick evidence-length check
evidence_lines = [ln for ln in resp.splitlines() if ln.lower().startswith("evidence:")]
joined = " ".join([ln.split("Evidence:",1)[1].strip() for ln in evidence_lines])
print("\nEvidence words:", word_count(joined))


Verdict: Supported
Evidence: “Policy requires review within 2 weeks.” (para 2)
Evidence: “Applies to onboarding and quarterly audits.” (para 3)

Evidence words: 16


## 3) Self-Ask (decompose via sub-questions)
**Goal:** Generate **2–4 sub-questions** with **≤12-word** answers, then one final answer in the required format.

In [4]:
resp = mock_model("self-ask with sub-questions and final")
print(resp)
# Enforce ≤12 words per A*
for ln in resp.splitlines():
    if "A" in ln:
        ans = ln.split("A",1)[1]
        w = word_count(ans)
        print(f"[len={w:>2}] {ln}")


Q1: Audience and tone? A1: Non-technical execs; plain, concise.
Q2: What is retrieval? A2: Finds relevant info to answer.
Q3: Why it helps? A3: Faster, fewer mistakes.
Final: Four bullets ≤60 words, exec tone.
[len= 9] Q1: Audience and tone? A1: Non-technical execs; plain, concise.
[len= 6] Q2: What is retrieval? A2: Finds relevant info to answer.
[len= 4] Q3: Why it helps? A3: Faster, fewer mistakes.


## 4) Contrastive CoT (2–3 options → pick with one-line why)
**Goal:** Request a few concise candidates, then **pick one** with a **one-line reason** tied to constraints.

In [5]:
resp = mock_model("contrastive options, then pick best with one-line why")
print(resp)
# Auto-pick check (simulate scoring by length + presence of 'policy')
lines = resp.splitlines()
opts = [ln for ln in lines if ln[:2] in ("A)", "B)", "C)")]
scores = [(i, (("policy" in ln.lower())*2 + (len(ln) < 120))) for i, ln in enumerate(opts)]
pick_i = max(range(len(scores)), key=lambda i: scores[i][1])
print("\nSimulated pick:", ["A","B","C"][pick_i], "scores:", scores)


A) Retrieval is like a smart search that shows only what matters.
B) Retrieval brings the **right policy** to your question in seconds.
C) Retrieval reduces noise by filtering to approved sources only.
Pick: B — Why: specific, executive-relevant, action-oriented.

Simulated pick: B scores: [(0, 1), (1, 3), (2, 1)]


## 5) Plan‑Then‑Solve (micro‑plan → format‑aligned answer)
**Goal:** Write a **2–3 line plan**, then produce the final answer *separately* in the required format & budget.

In [6]:
plan = mock_model("micro-plan then output")
print(plan)
final_bullets = "\n".join([
    "• Retrieval finds the most relevant information for your question.",
    "• It reduces noise and speeds up decisions with focused context.",
    "• Uses your approved sources (policies, docs, tickets).",
    "• Example: surfaces the right policy for a customer email."
])
ok, meta = verify_bullets_count_and_limit(final_bullets, expected_count=4, max_words_total=60)
print("\nFinal bullets:\n" + final_bullets)
print("\nVerifier (count/limit):", ok, meta)


Plan: define in one line → pick two benefits → add one example.
Output: bullets that follow the plan, ≤ word limit.

Final bullets:
• Retrieval finds the most relevant information for your question.
• It reduces noise and speeds up decisions with focused context.
• Uses your approved sources (policies, docs, tickets).
• Example: surfaces the right policy for a customer email.

Verifier (count/limit): True {'bullets': 4, 'total_words': 35}


## 6) Assumptions & Risk (≤30 words total)
**Goal:** List **1–2 assumptions** and **1 risk + mitigation** briefly.

In [7]:
line = "Assumptions: access to approved sources; users follow results. Risk: policy updates mid-cycle — Mitigation: version gate."
wc = word_count(line)
print(line, f"\n(words={wc})")
print("≤30 words:", wc <= 30)


Assumptions: access to approved sources; users follow results. Risk: policy updates mid-cycle — Mitigation: version gate. 
(words=16)
≤30 words: True


## 7) Lightweight Verifier (single rule)
**Goal:** After generating the final output, run **one rule** (structure + bounds + must/never).

In [8]:
rule = {
    "structure": {"bullets": 4},
    "bounds": {"total_words": 60},
    "tone": "exec_friendly"
}
print("Rule:", rule)
ok1, meta1 = verify_bullets_count_and_limit(final_bullets, expected_count=rule["structure"]["bullets"], max_words_total=rule["bounds"]["total_words"])
ok2 = verify_exec_tone(final_bullets)
status = "PASS" if (ok1 and ok2) else "FAIL"
print("Result:", status, "| details:", {**meta1, "exec_tone_ok": ok2})


Rule: {'structure': {'bullets': 4}, 'bounds': {'total_words': 60}, 'tone': 'exec_friendly'}
Result: PASS | details: {'bullets': 4, 'total_words': 35, 'exec_tone_ok': True}


## 8) ReAct Loop (glue demo — prompts only, no tools)
**Goal:** Show **Reason → Act → Observe → Repeat → Final → Verify** on the same retrieval example.

In [9]:
log = []

# Reason
plan = "Plan: define retrieval plainly → pick two exec benefits → add one example."
log.append(("Reason", plan))

# Act (make two options, pick best)
opt1 = "Retrieval finds the most relevant info fast, from your approved sources."
opt2 = "Retrieval brings the right policy to your inbox, cutting search time."
pick = opt2
log.append(("Act", f"Generated options; picked: {pick}"))

# Observe (evidence/checks)
obs = "Checks: 4 bullets planned; under 60 words; example included."
log.append(("Observe", obs))

# Final
final = "\n".join([
    "• Retrieval finds relevant information fast from approved sources.",
    "• Cuts noise and speeds decisions for busy teams.",
    "• Reduces risk by using current, vetted policies.",
    "• Example: answers a customer email with the correct policy."
])
log.append(("Final", final))

# Verify
ok, meta = verify_bullets_count_and_limit(final, 4, 60)
tone_ok = verify_exec_tone(final)
status = "PASS" if (ok and tone_ok) else "FAIL"
log.append(("Verify", f"{status} — {meta} — tone_ok={tone_ok}"))

for step, msg in log:
    print(f"== {step}\n{msg}\n")


== Reason
Plan: define retrieval plainly → pick two exec benefits → add one example.

== Act
Generated options; picked: Retrieval brings the right policy to your inbox, cutting search time.

== Observe
Checks: 4 bullets planned; under 60 words; example included.

== Final
• Retrieval finds relevant information fast from approved sources.
• Cuts noise and speeds decisions for busy teams.
• Reduces risk by using current, vetted policies.
• Example: answers a customer email with the correct policy.

== Verify
PASS — {'bullets': 4, 'total_words': 32} — tone_ok=True

